In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

url = "https://raw.githubusercontent.com/pandas-dev/pandas/main/pandas/tests/io/data/csv/tips.csv"
tips = pd.read_csv(url)
X = tips.drop(columns="tip")
y = tips["tip"]

In [2]:
# 1) train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [3]:
# 2) pipeline: čísla štandardizuj, kategórie preveď na 0/1 stĺpce, potom regresia
pre = ColumnTransformer([
    ("num", StandardScaler(), ["total_bill", "size"]),
    ("cat", OneHotEncoder(drop="first"), ["sex", "smoker", "day", "time"]),
])
model = make_pipeline(pre, LinearRegression())
model.fit(X_train, y_train)
print("MAE model:", mean_absolute_error(y_test, model.predict(X_test)))

baseline = DummyRegressor().fit(X_train, y_train)   # vždy tipne priemer
print("MAE baseline:", mean_absolute_error(y_test, baseline.predict(X_test)))

MAE model: 0.6974878153580388
MAE baseline: 1.0383552808384844


In [4]:
# 3) cross-validation
scores = -cross_val_score(model, X, y, cv=5, scoring="neg_mean_absolute_error")
print(scores, scores.mean(), scores.std())

[0.69769748 0.58141876 0.76853241 1.02048995 0.88319187] 0.7902660949084371 0.15119307611551372
